# Mermaid Visualizer for Team 04 Workflow
This notebook analyzes the files in the `PY` folder, extracts basic workflow metadata, and renders a Mermaid diagram directly inside VS Code.

## What it does
1. Loads local dependencies and HTML/JS visualization support.
2. Reads and lists the Python files in the `PY` folder.
3. Extracts states, nodes, and expected workflow routes.
4. Builds the Mermaid diagram with the requested visual style.
5. Renders the diagram inline inside the notebook.
6. Leaves a function ready to rebuild and re-render the diagram after changes in `PY`.

In [10]:
# 1. Load Mermaid dependencies and configure paths

from pathlib import Path
import importlib
import json
import re

import workflow_mermaid_utils as workflow_mermaid_utils
importlib.reload(workflow_mermaid_utils)

export_mermaid_png = workflow_mermaid_utils.export_mermaid_png
render_mermaid = workflow_mermaid_utils.render_mermaid

PY_DIR = Path(r"c:\Users\Arq. David Agudelo\Downloads\AIA26_Studio-team_04 (1)\AIA26_Studio-team_04\team_04\PY")
TEAM_ROOT = PY_DIR.parent
KEY_FILES = [
    "design_state.py",
    "design_workflow_graph.py",
    "design_routing.py",
    "central_reasoning_node.py",
    "design_action_nodes.py",
    "design_tool_node.py",
]
PREVIEW_PNG_PATH = TEAM_ROOT / "workflow_mermaid_preview.png"

print(f"PY_DIR configured as: {PY_DIR}")
print(f"Team root configured as: {TEAM_ROOT}")
print("Dependencies loaded. Mermaid renders with HTML + JS from a CDN and can export to PNG with Mermaid CLI or a local browser.")
print("workflow_mermaid_utils reloaded.")

PY_DIR configured as: c:\Users\Arq. David Agudelo\Downloads\AIA26_Studio-team_04 (1)\AIA26_Studio-team_04\team_04\PY
Team root configured as: c:\Users\Arq. David Agudelo\Downloads\AIA26_Studio-team_04 (1)\AIA26_Studio-team_04\team_04
Dependencies loaded. Mermaid renders with HTML + JS from a CDN and can export to PNG with Mermaid CLI or a local browser.
workflow_mermaid_utils reloaded.


## 2. Read and list the files in the `PY` folder
This cell scans the `PY` folder, detects `.py` files, and validates that the key files used by the diagram are available.

In [6]:
py_files = sorted(PY_DIR.glob("*.py"))
py_file_names = [path.name for path in py_files]
key_file_status = {name: (PY_DIR / name).exists() for name in KEY_FILES}

print(f"Python files detected: {len(py_file_names)}")
for name in py_file_names:
    print(f" - {name}")

print("\nKey diagram files:")
for name, exists in key_file_status.items():
    label = "OK" if exists else "MISSING"
    print(f" - {name}: {label}")

Python files detected: 13
 - central_reasoning_node.py
 - design_action_nodes.py
 - design_config.py
 - design_main.py
 - design_main_test.py
 - design_registry.py
 - design_routing.py
 - design_state.py
 - design_tool_node.py
 - design_workflow_graph.py
 - mcp_client.py
 - tool_node.py
 - workflow_mermaid_utils.py

Key diagram files:
 - design_state.py: OK
 - design_workflow_graph.py: OK
 - design_routing.py: OK
 - central_reasoning_node.py: OK
 - design_action_nodes.py: OK
 - design_tool_node.py: OK


## 3. Extract nodes, edges, and metadata from the Python files
This cell performs lightweight parsing of `TypedDict`, registered graph nodes, and workflow routers to preserve basic traceability between files and components.

In [3]:
def extract_workflow_metadata(file_texts: dict[str, str]) -> dict[str, object]:
    state_text = file_texts.get("design_state.py", "")
    action_text = file_texts.get("design_action_nodes.py", "")
    graph_text = file_texts.get("design_workflow_graph.py", "")
    routing_text = file_texts.get("design_routing.py", "")
    tool_text = file_texts.get("design_tool_node.py", "")
    central_text = file_texts.get("central_reasoning_node.py", "")

    states = re.findall(r"class\s+(\w+)\(TypedDict\):", state_text)
    graph_nodes = re.findall(r'graph\.add_node\("([^"]+)"', graph_text)
    action_creators = re.findall(r"def\s+create_([a-z_]+)_node", action_text)
    routers = re.findall(r"def\s+(create_route_after_[a-z_]+)\(", routing_text)

    return {
        "states": states,
        "graph_nodes": graph_nodes,
        "action_creators": action_creators,
        "routers": routers,
        "has_design_tool_node": "def create_design_tool_node" in tool_text,
        "has_central_reasoning": "def create_central_reasoning_node" in central_text,
    }

file_text = {path.name: path.read_text(encoding="utf-8") for path in py_files}
workflow_metadata = extract_workflow_metadata(file_text)

print(json.dumps(workflow_metadata, indent=2, ensure_ascii=False))

{
  "states": [
    "DesignWorkflowState",
    "SceneState"
  ],
  "graph_nodes": [
    "central_reason",
    "suggestion",
    "evaluation",
    "optimization",
    "explanation",
    "visualization",
    "constraint_check",
    "user_feedback",
    "design_tool",
    "finish"
  ],
  "action_creators": [
    "suggestion",
    "evaluation",
    "optimization",
    "explanation",
    "visualization",
    "constraint_check",
    "user_feedback"
  ],
  "routers": [
    "create_route_after_central_reasoning",
    "create_route_after_action_node",
    "create_route_after_constraint_check",
    "create_route_after_tool_execution",
    "create_route_after_user_feedback"
  ],
  "has_design_tool_node": true,
  "has_central_reasoning": true
}


## 4. Build the Mermaid diagram with the `init` block and the requested style
This cell generates the final Mermaid string from a template that references the key workflow files.

In [7]:
diagram_files = {
    "state": "design_state.py",
    "graph": "design_workflow_graph.py",
    "routing": "design_routing.py",
    "central": "central_reasoning_node.py",
    "action": "design_action_nodes.py",
    "tool": "design_tool_node.py",
}

def build_mermaid_diagram(file_map: dict[str, str]) -> str:
    template = '''%%{init: {
  "theme": "base",
  "flowchart": {
    "curve": "bumpX",
    "nodeSpacing": 120,
    "rankSpacing": 135,
    "padding": 34
  },
  "themeVariables": {
    "background": "#ffffff",
    "primaryTextColor": "#111111",
    "lineColor": "#6c757d",
    "fontSize": "16px",
    "fontFamily": "Segoe UI, Arial, sans-serif"
  }
}}%%

flowchart LR

    STATE[(DesignWorkflowState<br/>__STATE_FILE__)]
    CONTROL[[edges: __GRAPH_FILE__<br/>routes: __ROUTING_FILE__]]

    START([START<br/>__GRAPH_FILE__])

    subgraph R["1. Reasoning"]
        direction TB
        CR[central_reason<br/>__CENTRAL_FILE__]
    end

    subgraph A["2. Action Nodes"]
        direction TB
        SG[suggestion<br/>__ACTION_FILE__]
        EV[evaluation<br/>__ACTION_FILE__]
        OP[optimization<br/>__ACTION_FILE__]
        EX[explanation<br/>__ACTION_FILE__]
        VZ[visualization<br/>__ACTION_FILE__]
        UF[user_feedback<br/>__ACTION_FILE__]
    end

    subgraph X["3. Execution / Validation"]
        direction TB
        DT[design_tool<br/>__TOOL_FILE__]
        CC[constraint_check<br/>__ACTION_FILE__]
    end

    subgraph F["4. Finish"]
        direction TB
        FN[finish<br/>__GRAPH_FILE__]
        ENDNODE([END<br/>__GRAPH_FILE__])
    end

    START ==> CR

    CR ==>|suggest| SG
    CR ==>|evaluate| EV
    CR ==>|optimize| OP
    CR ==>|explain| EX
    CR ==>|visualize| VZ
    CR -.->|ask_user| UF
    CR ==>|final| FN

    SG -->|tool| DT
    EV -->|tool| DT
    OP -->|tool| DT
    EX -->|tool| DT
    VZ -->|tool| DT

    SG -->|check| CC
    EV -->|check| CC
    OP -->|check| CC
    EX -->|check| CC
    VZ -->|check| CC

    SG -.->|iterate| CR
    EV -.->|iterate| CR
    OP -.->|iterate| CR
    EX -.->|iterate| CR
    VZ -.->|iterate| CR

    DT -.->|results| CR
    DT ==>|max iterations| FN

    CC -.->|continue| CR
    CC ==>|finish| FN

    UF -.->|feedback| CR
    FN ==> ENDNODE

    STATE -. shared state .-> CR
    STATE -. shared state .-> DT
    STATE -. shared state .-> CC

    CONTROL -. controls .-> CR
    CONTROL -. controls .-> DT
    CONTROL -. controls .-> CC
    CONTROL -. controls .-> FN

    classDef state fill:#f1f3f5,stroke:#868e96,stroke-width:1.8px,color:#111111;
    classDef control fill:#fff9db,stroke:#adb5bd,stroke-width:1.8px,stroke-dasharray:4 4,color:#111111;
    classDef startEnd fill:#ffe3e3,stroke:#c92a2a,stroke-width:2.8px,color:#111111;
    classDef reason fill:#fff3bf,stroke:#f08c00,stroke-width:3px,color:#111111;
    classDef action fill:#dbeafe,stroke:#2563eb,stroke-width:2.2px,color:#111111;
    classDef feedback fill:#e6fcf5,stroke:#0ca678,stroke-width:2.2px,color:#111111;
    classDef tool fill:#d3f9d8,stroke:#2b8a3e,stroke-width:2.4px,color:#111111;
    classDef check fill:#efe1ff,stroke:#7048e8,stroke-width:2.4px,color:#111111;
    classDef finish fill:#ffe8cc,stroke:#e67700,stroke-width:2.6px,color:#111111;

    class STATE state;
    class CONTROL control;
    class START,ENDNODE startEnd;
    class CR reason;
    class SG,EV,OP,EX,VZ action;
    class UF feedback;
    class DT tool;
    class CC check;
    class FN finish;

    style R fill:#fffdf4,stroke:#f1d8a8,stroke-width:1.2px,color:#111111
    style A fill:#f8fbff,stroke:#bfd7ff,stroke-width:1.2px,color:#111111
    style X fill:#f4fff7,stroke:#b7e4c7,stroke-width:1.2px,color:#111111
    style F fill:#fff5f5,stroke:#ffc9c9,stroke-width:1.2px,color:#111111

    %% 0 start -> central_reason
    linkStyle 0 stroke:#495057,stroke-width:3.2px;

    %% 1-7 central reasoning routes
    linkStyle 1 stroke:#f08c00,stroke-width:2.9px;
    linkStyle 2 stroke:#f08c00,stroke-width:2.9px;
    linkStyle 3 stroke:#f08c00,stroke-width:2.9px;
    linkStyle 4 stroke:#f08c00,stroke-width:2.9px;
    linkStyle 5 stroke:#f08c00,stroke-width:2.9px;
    linkStyle 6 stroke:#0ca678,stroke-width:2.2px,stroke-dasharray:7 5;
    linkStyle 7 stroke:#c92a2a,stroke-width:3px;

    %% 8-12 actions -> tool
    linkStyle 8 stroke:#2b8a3e,stroke-width:2.3px;
    linkStyle 9 stroke:#2b8a3e,stroke-width:2.3px;
    linkStyle 10 stroke:#2b8a3e,stroke-width:2.3px;
    linkStyle 11 stroke:#2b8a3e,stroke-width:2.3px;
    linkStyle 12 stroke:#2b8a3e,stroke-width:2.3px;

    %% 13-17 actions -> constraint_check
    linkStyle 13 stroke:#7048e8,stroke-width:2.3px;
    linkStyle 14 stroke:#7048e8,stroke-width:2.3px;
    linkStyle 15 stroke:#7048e8,stroke-width:2.3px;
    linkStyle 16 stroke:#7048e8,stroke-width:2.3px;
    linkStyle 17 stroke:#7048e8,stroke-width:2.3px;

    %% 18-22 action loops back to reasoning
    linkStyle 18 stroke:#6c757d,stroke-width:2px,stroke-dasharray:7 5;
    linkStyle 19 stroke:#6c757d,stroke-width:2px,stroke-dasharray:7 5;
    linkStyle 20 stroke:#6c757d,stroke-width:2px,stroke-dasharray:7 5;
    linkStyle 21 stroke:#6c757d,stroke-width:2px,stroke-dasharray:7 5;
    linkStyle 22 stroke:#6c757d,stroke-width:2px,stroke-dasharray:7 5;

    %% 23-28 tool / validation / finish
    linkStyle 23 stroke:#2b8a3e,stroke-width:2.2px,stroke-dasharray:7 5;
    linkStyle 24 stroke:#c92a2a,stroke-width:3px;
    linkStyle 25 stroke:#7048e8,stroke-width:2.2px,stroke-dasharray:7 5;
    linkStyle 26 stroke:#c92a2a,stroke-width:3px;
    linkStyle 27 stroke:#0ca678,stroke-width:2px,stroke-dasharray:7 5;
    linkStyle 28 stroke:#c92a2a,stroke-width:3.2px;

    %% 29-31 state relations
    linkStyle 29 stroke:#868e96,stroke-width:1.6px,stroke-dasharray:4 4;
    linkStyle 30 stroke:#868e96,stroke-width:1.6px,stroke-dasharray:4 4;
    linkStyle 31 stroke:#868e96,stroke-width:1.6px,stroke-dasharray:4 4;

    %% 32-35 control relations
    linkStyle 32 stroke:#adb5bd,stroke-width:1.6px,stroke-dasharray:4 4;
    linkStyle 33 stroke:#adb5bd,stroke-width:1.6px,stroke-dasharray:4 4;
    linkStyle 34 stroke:#adb5bd,stroke-width:1.6px,stroke-dasharray:4 4;
    linkStyle 35 stroke:#adb5bd,stroke-width:1.6px,stroke-dasharray:4 4;
'''

    replacements = {
        "__STATE_FILE__": file_map["state"],
        "__GRAPH_FILE__": file_map["graph"],
        "__ROUTING_FILE__": file_map["routing"],
        "__CENTRAL_FILE__": file_map["central"],
        "__ACTION_FILE__": file_map["action"],
        "__TOOL_FILE__": file_map["tool"],
    }

    for placeholder, value in replacements.items():
        template = template.replace(placeholder, value)

    return template

mermaid_source = build_mermaid_diagram(diagram_files)
print(f"Workflow Mermaid built with {len(mermaid_source.splitlines())} lines.")

Workflow Mermaid built with 173 lines.


## 5. Render the Mermaid diagram inline in VS Code and export it to PNG
The next cell generates the preview HTML, tries to draw the diagram inline, and also exports a PNG to the `team_04` root folder using Chrome or Edge in headless mode first. If that fails, it falls back to Mermaid CLI.

In [ ]:
preview_html_path = render_mermaid(mermaid_source, PY_DIR)
preview_png_path = export_mermaid_png(
    mermaid_source,
    PY_DIR,
    output_path=PREVIEW_PNG_PATH,
)

## 6. Validate the rendered result and rebuild the diagram after changes in `PY`
The final cell checks for missing files, detects `.py` files that are not yet reflected in the diagram, and prepares a rebuild function.

In [9]:
DIAGRAM_REFERENCED_FILES = {
    diagram_files["state"],
    diagram_files["graph"],
    diagram_files["routing"],
    diagram_files["central"],
    diagram_files["action"],
    diagram_files["tool"],
}

def rebuild_and_render() -> str:
    global py_files, py_file_names, key_file_status, file_text, workflow_metadata, mermaid_source
    global preview_html_path, preview_png_path

    py_files = sorted(PY_DIR.glob("*.py"))
    py_file_names = [path.name for path in py_files]
    key_file_status = {name: (PY_DIR / name).exists() for name in KEY_FILES}
    file_text = {path.name: path.read_text(encoding="utf-8") for path in py_files}
    workflow_metadata = extract_workflow_metadata(file_text)
    mermaid_source = build_mermaid_diagram(diagram_files)

    missing_files = [name for name, exists in key_file_status.items() if not exists]
    unreferenced_py_files = sorted(set(py_file_names) - DIAGRAM_REFERENCED_FILES)

    preview_html_path = render_mermaid(mermaid_source, PY_DIR)
    preview_png_path = export_mermaid_png(
        mermaid_source,
        PY_DIR,
        output_path=PREVIEW_PNG_PATH,
    )

    validation_report = {
        "missing_key_files": missing_files,
        "states_found": workflow_metadata["states"],
        "graph_nodes_found": workflow_metadata["graph_nodes"],
        "routers_found": workflow_metadata["routers"],
        "unreferenced_python_files": unreferenced_py_files,
        "preview_html_path": str(preview_html_path),
        "preview_png_path": str(preview_png_path) if preview_png_path else None,
    }

    print(json.dumps(validation_report, indent=2, ensure_ascii=False))
    return mermaid_source

validation_preview = {
    "missing_key_files": [name for name, exists in key_file_status.items() if not exists],
    "unreferenced_python_files": sorted(set(py_file_names) - DIAGRAM_REFERENCED_FILES),
}

print("Run rebuild_and_render() after changes in the PY folder to regenerate the HTML, PNG, and inline diagram.")
print(json.dumps(validation_preview, indent=2, ensure_ascii=False))

Run rebuild_and_render() after changes in the PY folder to regenerate the HTML, PNG, and inline diagram.
{
  "missing_key_files": [],
  "unreferenced_python_files": [
    "design_config.py",
    "design_main.py",
    "design_main_test.py",
    "design_registry.py",
    "mcp_client.py",
    "tool_node.py",
    "workflow_mermaid_utils.py"
  ]
}


## Quick usage
- Run cells 2, 4, 6, 8, 10, and 12 in order to rebuild the full workflow.
- Cell 10 generates `workflow_mermaid_preview.html`, exports `workflow_mermaid_preview.png` to the `team_04` root folder, and shows the rendered result inline when the export succeeds.
- If you modify files in `PY`, run `rebuild_and_render()` in the last cell.
- If `npx` needs to install Mermaid CLI for the first time, the PNG export may take several minutes; run cell 10 again if the first attempt times out.
- If the CDN is blocked, the final string will still be available in `mermaid_source` so you can reuse it in another Mermaid tool.